# Sistema de recomendación para clientes sin historial (cold start) — H&M Argentina
## Sprint 1 — Análisis Exploratorio de Datos (EDA) sobre la muestra p10

---

## Índice

1. [Introducción y Contexto de Negocio](#1)
2. [Importación de Librerías](#2)
3. [Carga y Exploración Inicial de Datos](#3)
4. [Diccionario de Variables](#4)
5. [Calidad de Datos e Integridad entre Tablas](#5)
6. [Transacciones: Análisis Temporal](#6)
7. [Transacciones: Precio, Clientes y Artículos](#7)
8. [Clientes](#8)
9. [Artículos](#9)
10. [Análisis Cruzado](#10)
11. [Partición Temporal Train / Test](#11)
12. [Clientes Fríos y Cobertura de Artículos](#12)
13. [Hallazgos Consolidados y Decisiones Pendientes](#13)

<a id="1"></a>
## 1. Introducción y Contexto de Negocio

H&M abrirá su primera tienda en Argentina en 2027 y no cuenta con datos locales. Se diseña un recomendador para **clientes sin historial** (*cold start*), con dos módulos:

- **Módulo 1 (principal):** recomendaciones para clientes fríos. KPI principal: MAP@12; secundario: Recall@12.
- **Módulo 2 (extensión):** "Completá el look". KPIs: cobertura y HitRate@K.

Este notebook documenta el EDA sobre la **muestra p10**: 10 % de los clientes, seleccionados con hash determinístico de `customer_id`, restringida al canal online (`sales_channel_id = 2`) y con todo el historial online de cada cliente conservado.

**Principios metodológicos del análisis:**
- Solo se utilizan los datos reales de los archivos; ninguna cifra se escribe a mano, todas se calculan en las celdas.
- Las decisiones que dependen de los datos (corte temporal, tratamiento de edad faltante) se toman explorando únicamente el conjunto de entrenamiento, para evitar *data leakage*. Las secciones 5 a 10 describen la muestra completa; la sección 11 define el corte y la sección 12 compara train contra test.

**Limitaciones a declarar:** las estaciones del dataset corresponden al hemisferio norte (invertidas respecto de Argentina), el mercado europeo se usa como aproximación, no se consideran promociones y la evaluación es *offline*.

<a id="2"></a>
## 2. Importación de Librerías

In [ ]:
# ===== MANIPULACIÓN DE DATOS =====
import pandas as pd
import numpy as np
from pathlib import Path

# ===== VISUALIZACIÓN =====
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ===== UTILIDADES =====
import warnings
warnings.filterwarnings('ignore')

# Estilo general de los gráficos
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlepad'] = 14

# Colores del notebook
AZUL = '#2E86AB'
ROJO = '#E63946'
GRIS = '#8D99AE'

# Formato numérico de las tablas
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 40
print('Librerías cargadas.')

<a id="3"></a>
## 3. Carga y Exploración Inicial de Datos

Los archivos se leen desde `dataset/p10/` (ruta relativa a la carpeta del notebook). `article_id` se conserva como **texto de 10 caracteres** para no perder los ceros iniciales; `t_dat` se convierte a fecha.

In [ ]:
# Ruta de los datos (relativa a la ubicación de este notebook)
RUTA = Path('dataset/p10')

# Lectura de las tres tablas en formato parquet
transactions = pd.read_parquet(RUTA / 'transactions.parquet')
customers    = pd.read_parquet(RUTA / 'customers.parquet')
articles     = pd.read_parquet(RUTA / 'articles.parquet')

# article_id como texto de 10 caracteres (preserva los ceros iniciales)
transactions['article_id'] = transactions['article_id'].astype(str).str.zfill(10)
articles['article_id']     = articles['article_id'].astype(str).str.zfill(10)

# t_dat como fecha
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print('DIMENSIONES')
print('=' * 70)
for nombre, tabla in [('transactions', transactions), ('customers', customers), ('articles', articles)]:
    print(f'{nombre:13s}: {tabla.shape[0]:>10,} filas x {tabla.shape[1]:>2} columnas')

In [ ]:
# Primeras filas de cada tabla
display(transactions.head())
display(customers.head())
display(articles.head(3).T)

In [ ]:
# Tipos de dato y memoria de cada tabla
for nombre, tabla in [('transactions', transactions), ('customers', customers), ('articles', articles)]:
    print(f'--- {nombre} ---')
    tabla.info(memory_usage='deep')
    print()

<a id="4"></a>
## 4. Diccionario de Variables

Descripción de las columnas según la documentación pública del dataset (Kaggle, *H&M Personalized Fashion Recommendations*). El precio está escalado: **no representa dinero real**.

In [ ]:
# Diccionario resumido de las tres tablas
diccionario = pd.DataFrame([
    ('transactions', 't_dat',            'Fecha de la compra'),
    ('transactions', 'customer_id',      'Identificador del cliente (hash)'),
    ('transactions', 'article_id',       'Identificador del artículo (texto de 10 caracteres)'),
    ('transactions', 'price',            'Precio pagado (escalado, no es dinero real)'),
    ('transactions', 'sales_channel_id', 'Canal de venta (2 = online; inferencia, no documentación oficial)'),
    ('customers', 'FN',                  'Indicador de suscripción a Fashion News (1 / nulo)'),
    ('customers', 'Active',              'Indicador de cliente activo en comunicaciones (1 / nulo)'),
    ('customers', 'club_member_status',  'Estado en el club de miembros'),
    ('customers', 'fashion_news_frequency', 'Frecuencia de envío de novedades'),
    ('customers', 'age',                 'Edad del cliente'),
    ('customers', 'postal_code',         'Código postal (hash)'),
    ('articles', 'article_id',           'Identificador único del artículo'),
    ('articles', 'product_code',         'Código del producto (agrupa variantes de color)'),
    ('articles', 'product_type_name',    'Tipo de producto'),
    ('articles', 'product_group_name',   'Grupo de producto'),
    ('articles', 'colour_group_name',    'Grupo de color'),
    ('articles', 'department_name',      'Departamento'),
    ('articles', 'index_group_name',     'Grupo de índice (Ladieswear, Menswear, etc.)'),
    ('articles', 'section_name',         'Sección'),
    ('articles', 'garment_group_name',   'Grupo de prenda'),
    ('articles', 'detail_desc',          'Descripción textual del artículo'),
], columns=['tabla', 'variable', 'descripcion'])

display(diccionario)
print('\nColumnas de articles no listadas arriba (códigos numéricos y variantes de nombre):')
print([c for c in articles.columns if c not in diccionario['variable'].values])

<a id="5"></a>
## 5. Calidad de Datos e Integridad entre Tablas

Se verifican valores nulos, duplicados, unicidad de claves e integridad referencial (todo `customer_id` y `article_id` de `transactions` debe existir en su tabla maestra).

In [ ]:
# --- Valores nulos por tabla ---
def reporte_nulos(tabla, nombre):
    """Muestra cantidad y porcentaje de nulos por columna (solo las que tienen)."""
    n = tabla.isnull().sum()
    rep = pd.DataFrame({'n_nulos': n, 'pct_nulos': (n / len(tabla) * 100).round(2)})
    rep = rep[rep['n_nulos'] > 0]
    print(f'VALORES NULOS — {nombre}')
    print('=' * 70)
    if rep.empty:
        print('No se encontraron valores nulos.')
    else:
        display(rep)
    print()

reporte_nulos(transactions, 'transactions')
reporte_nulos(customers, 'customers')
reporte_nulos(articles, 'articles')

In [ ]:
# --- Duplicados y unicidad de claves ---
print('DUPLICADOS Y CLAVES')
print('=' * 70)
print(f'customers: customer_id duplicados = {customers["customer_id"].duplicated().sum():,}')
print(f'articles : article_id duplicados  = {articles["article_id"].duplicated().sum():,}')

# En transactions las filas repetidas NO se eliminan: representan unidades múltiples de la misma compra
dup_trans = transactions.duplicated().sum()
print(f'transactions: filas completamente repetidas = {dup_trans:,} ({dup_trans / len(transactions) * 100:.2f} %)')
print('  -> Se conservan: corresponden a unidades múltiples del mismo artículo en el mismo día.')

In [ ]:
# --- Integridad referencial ---
clientes_trans    = set(transactions['customer_id'].unique())
clientes_maestro  = set(customers['customer_id'])
articulos_trans   = set(transactions['article_id'].unique())
articulos_maestro = set(articles['article_id'])

print('INTEGRIDAD REFERENCIAL')
print('=' * 70)
print(f'Clientes en transactions que NO están en customers : {len(clientes_trans - clientes_maestro):,}')
print(f'Clientes en customers sin ninguna transacción      : {len(clientes_maestro - clientes_trans):,}')
print(f'Artículos en transactions que NO están en articles : {len(articulos_trans - articulos_maestro):,}')
sin_venta = len(articulos_maestro - articulos_trans)
print(f'Artículos del catálogo sin ninguna venta en la muestra: {sin_venta:,} ({sin_venta / len(articulos_maestro) * 100:.1f} % del catálogo)')

# Formato de article_id y canal
print(f'\nLargo de article_id en transactions: {transactions["article_id"].str.len().unique().tolist()}')
print(f'Valores de sales_channel_id        : {transactions["sales_channel_id"].unique().tolist()}')

<a id="6"></a>
## 6. Transacciones: Análisis Temporal

Se describe la actividad diaria, mensual y semanal. El objetivo es detectar días incompletos, estacionalidad y períodos atípicos antes de fijar el corte train/test.

In [ ]:
# Rango de fechas y cobertura de calendario
fecha_min, fecha_max = transactions['t_dat'].min(), transactions['t_dat'].max()
dias_calendario = (fecha_max - fecha_min).days + 1
dias_con_ventas = transactions['t_dat'].nunique()

print('RANGO TEMPORAL')
print('=' * 70)
print(f'Primera fecha: {fecha_min.date()}')
print(f'Última fecha : {fecha_max.date()}')
print(f'Días de calendario: {dias_calendario:,} | días con al menos una venta: {dias_con_ventas:,}')

# Conteo diario (reindexado para mostrar los días sin ventas como 0)
diario = (transactions.groupby('t_dat').size()
          .reindex(pd.date_range(fecha_min, fecha_max), fill_value=0)
          .rename('transacciones'))
print(f'Días sin ventas: {(diario == 0).sum()}')
print(f'Promedio diario: {diario.mean():,.0f} | mediana: {diario.median():,.0f} | máximo: {diario.max():,} ({diario.idxmax().date()})')

In [ ]:
# Serie diaria de transacciones
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(diario.index, diario.values, color=AZUL, linewidth=0.9)
ax.set_title('Transacciones por día (muestra p10, canal online)')
ax.set_ylabel('Cantidad de transacciones')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Serie mensual: transacciones, clientes activos y artículos distintos
trans_mes = transactions.assign(mes=transactions['t_dat'].dt.to_period('M'))
mensual = trans_mes.groupby('mes').agg(transacciones=('article_id', 'size'),
                                       clientes_activos=('customer_id', 'nunique'),
                                       articulos_distintos=('article_id', 'nunique'),
                                       dias_con_datos=('t_dat', 'nunique'))
# Transacciones por día con datos (controla los meses incompletos)
mensual['trans_por_dia'] = (mensual['transacciones'] / mensual['dias_con_datos']).round(0)
display(mensual)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
mensual['transacciones'].plot(kind='bar', ax=axes[0], color=AZUL, width=0.8)
axes[0].set_title('Transacciones por mes')
axes[0].set_xlabel('')
mensual['trans_por_dia'].plot(kind='bar', ax=axes[1], color=ROJO, width=0.8)
axes[1].set_title('Transacciones por día, promedio de cada mes')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Patrón semanal: transacciones por día de la semana
dias_es = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
por_dow = transactions.groupby(transactions['t_dat'].dt.dayofweek).size()
por_dow.index = dias_es

fig, ax = plt.subplots(figsize=(9, 5))
por_dow.plot(kind='bar', ax=ax, color=AZUL, width=0.6)
ax.set_title('Transacciones por día de la semana')
ax.set_xlabel('')
plt.xticks(rotation=0)
for i, v in enumerate(por_dow):
    ax.text(i, v, f'{v / por_dow.sum() * 100:.1f} %', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

<a id="7"></a>
## 7. Transacciones: Precio, Clientes y Artículos

In [ ]:
# --- Precio ---
print('PRECIO (escalado, no es dinero real)')
print('=' * 70)
display(transactions['price'].describe(percentiles=[.01, .25, .5, .75, .99]).to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(transactions['price'], bins=60, color=AZUL, ax=axes[0])
axes[0].set_title('Distribución del precio')
sns.histplot(np.log10(transactions['price']), bins=60, color=ROJO, ax=axes[1])
axes[1].set_title('Distribución del precio (log10)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Compras por cliente ---
compras_cliente = transactions.groupby('customer_id').size()
print('TRANSACCIONES POR CLIENTE')
print('=' * 70)
display(compras_cliente.describe(percentiles=[.1, .25, .5, .75, .9, .99]).to_frame().T)
print(f'Clientes con una sola transacción   : {(compras_cliente == 1).sum():,} ({(compras_cliente == 1).mean() * 100:.1f} %)')
print(f'Clientes con 5 o menos transacciones: {(compras_cliente <= 5).sum():,} ({(compras_cliente <= 5).mean() * 100:.1f} %)')

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(compras_cliente.clip(upper=compras_cliente.quantile(0.99)), bins=60, color=AZUL, ax=ax)
ax.set_title('Transacciones por cliente (truncado en el percentil 99)')
ax.set_xlabel('Transacciones por cliente')
plt.tight_layout()
plt.show()

In [ ]:
# --- Popularidad de artículos (cola larga) ---
ventas_art = transactions.groupby('article_id').size().sort_values(ascending=False)
acum = ventas_art.cumsum() / ventas_art.sum()

print('POPULARIDAD DE ARTÍCULOS')
print('=' * 70)
print(f'Artículos con al menos una venta: {len(ventas_art):,}')
for k in [12, 100, 1000, 5000]:
    print(f'Top {k:>5,} artículos concentran {acum.iloc[k - 1] * 100:5.1f} % de las ventas')
print(f'Artículos con una sola venta: {(ventas_art == 1).sum():,} ({(ventas_art == 1).mean() * 100:.1f} % de los vendidos)')
n50 = int((acum < 0.5).sum() + 1)
print(f'Artículos necesarios para cubrir el 50 % de las ventas: {n50:,}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(np.arange(1, len(ventas_art) + 1), ventas_art.values, color=AZUL)
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_title('Ventas por artículo (ranking, escala log-log)')
axes[0].set_xlabel('Ranking del artículo'); axes[0].set_ylabel('Ventas')
axes[1].plot(np.arange(1, len(acum) + 1) / len(acum) * 100, acum.values * 100, color=ROJO)
axes[1].set_title('Concentración acumulada de ventas')
axes[1].set_xlabel('% de artículos (ordenados por ventas)'); axes[1].set_ylabel('% acumulado de ventas')
plt.tight_layout()
plt.show()

<a id="8"></a>
## 8. Clientes

In [ ]:
# Variables categóricas de customers (los nulos se cuentan como categoría)
for col in ['club_member_status', 'fashion_news_frequency', 'FN', 'Active']:
    print(f'--- {col} ---')
    display(customers[col].value_counts(dropna=False).to_frame('n').assign(pct=lambda d: (d['n'] / d['n'].sum() * 100).round(2)))

In [ ]:
# Edad: descriptivos, faltantes y valores raros
print('EDAD')
print('=' * 70)
display(customers['age'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame().T)
n_na = customers['age'].isnull().sum()
print(f'Edad faltante: {n_na:,} ({n_na / len(customers) * 100:.2f} %)')
print(f'Edad < 16: {(customers["age"] < 16).sum():,} | edad > 90: {(customers["age"] > 90).sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customers['age'].dropna(), bins=range(14, 100, 2), color=AZUL, ax=axes[0])
axes[0].set_title('Distribución de edad')
sns.boxplot(x=customers['age'].dropna(), color=AZUL, ax=axes[1])
axes[1].set_title('Edad (boxplot)')
plt.tight_layout()
plt.show()

In [ ]:
# Outliers de edad por IQR
def detectar_outliers_iqr(serie):
    """Cuenta los valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((serie < inf) | (serie > sup)).sum()
    return inf, sup, n, n / len(serie) * 100

inf, sup, n, pct = detectar_outliers_iqr(customers['age'].dropna())
print(f'Edad — límites IQR: [{inf:.1f}, {sup:.1f}] | outliers: {n:,} ({pct:.2f} %)')

In [ ]:
# Actividad de compra según perfil: ¿los clientes sin edad compran distinto?
compras_df = compras_cliente.rename('n_compras').reset_index()
perfil = customers.merge(compras_df, on='customer_id', how='left')
perfil['edad_faltante'] = perfil['age'].isnull()

print('COMPRAS POR CLIENTE SEGÚN EDAD FALTANTE')
print('=' * 70)
display(perfil.groupby('edad_faltante')['n_compras'].describe())

print('COMPRAS POR CLIENTE SEGÚN club_member_status')
display(perfil.groupby('club_member_status')['n_compras'].describe())

<a id="9"></a>
## 9. Artículos

In [ ]:
# Cardinalidad de las variables de catálogo
cols_cat = ['product_group_name', 'product_type_name', 'colour_group_name', 'department_name',
            'index_group_name', 'section_name', 'garment_group_name']
card = pd.DataFrame({'categorias_distintas': [articles[c].nunique() for c in cols_cat]}, index=cols_cat)
display(card)

In [ ]:
# Catálogo vs. ventas por grupo de índice y grupo de producto
trans_art = transactions.merge(articles[['article_id', 'index_group_name', 'product_group_name',
                                         'product_type_name', 'colour_group_name']],
                               on='article_id', how='left')

def catalogo_vs_ventas(col):
    """Compara el % del catálogo con el % de las ventas de cada categoría."""
    cat = articles[col].value_counts(normalize=True) * 100
    ven = trans_art[col].value_counts(normalize=True) * 100
    return pd.DataFrame({'pct_catalogo': cat, 'pct_ventas': ven}).fillna(0).sort_values('pct_ventas', ascending=False)

for col in ['index_group_name', 'product_group_name']:
    print(f'--- {col}: % del catálogo vs % de las ventas ---')
    display(catalogo_vs_ventas(col).round(2))

In [ ]:
# Top 15 tipos de producto por ventas
top_tipos = trans_art['product_type_name'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
top_tipos.sort_values().plot(kind='barh', ax=ax, color=AZUL)
ax.set_title('Top 15 tipos de producto por cantidad de ventas')
ax.set_xlabel('Transacciones')
plt.tight_layout()
plt.show()

In [ ]:
# Calidad de las descripciones y nombres
print('CALIDAD DE LA DESCRIPCIÓN')
print('=' * 70)
print(f'detail_desc nulos : {articles["detail_desc"].isnull().sum():,} ({articles["detail_desc"].isnull().mean() * 100:.2f} %)')
print(f'prod_name nulos   : {articles["prod_name"].isnull().sum():,}')
print(f'Largo medio de detail_desc: {articles["detail_desc"].dropna().str.len().mean():.0f} caracteres')
print(f'Productos (product_code) distintos: {articles["product_code"].nunique():,} | artículos (variantes): {len(articles):,}')

<a id="10"></a>
## 10. Análisis Cruzado

Relación entre perfil del cliente y categorías de producto, y estacionalidad por categoría.

In [ ]:
# Mix de compras por grupo de índice, según tramo de edad
trans_cli = trans_art.merge(customers[['customer_id', 'age']], on='customer_id', how='left')
trans_cli['tramo_edad'] = pd.cut(trans_cli['age'], bins=[0, 20, 25, 30, 40, 50, 60, 120],
                                 labels=['<=20', '21-25', '26-30', '31-40', '41-50', '51-60', '>60'])

tabla_edad = pd.crosstab(trans_cli['tramo_edad'], trans_cli['index_group_name'], normalize='index') * 100
print('% DE LAS COMPRAS POR GRUPO DE ÍNDICE, SEGÚN TRAMO DE EDAD')
print('=' * 70)
display(tabla_edad.round(1))

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.heatmap(tabla_edad, annot=True, fmt='.1f', cmap='Blues', ax=ax, cbar_kws={'label': '% de compras del tramo'})
ax.set_title('Mix de compras por tramo de edad')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Estacionalidad: participación mensual de los principales grupos de producto
trans_art['mes'] = trans_art['t_dat'].dt.to_period('M').astype(str)
top_grupos = trans_art['product_group_name'].value_counts().head(6).index
mix_mes = (trans_art[trans_art['product_group_name'].isin(top_grupos)]
           .groupby(['mes', 'product_group_name']).size().unstack(fill_value=0))
mix_mes_pct = mix_mes.div(trans_art.groupby('mes').size(), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 5.5))
mix_mes_pct.plot(ax=ax, linewidth=1.8)
ax.set_title('Participación mensual de los principales grupos de producto (% de transacciones del mes)')
ax.set_ylabel('% de transacciones'); ax.set_xlabel('')
ax.legend(title='', fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Precio medio por grupo de índice y por grupo de producto
display(trans_art.groupby('index_group_name')['price'].agg(['count', 'mean', 'median']).sort_values('count', ascending=False))
display(trans_art.groupby('product_group_name')['price'].agg(['count', 'mean', 'median']).sort_values('count', ascending=False).head(10))

<a id="11"></a>
## 11. Partición Temporal Train / Test

La partición es **temporal, nunca al azar**: el conjunto de prueba son los **últimos 7 días completos**. El último día del dataset es el 2020-09-22 y el corte provisorio de trabajo fue el 2020-09-15, que incluiría 8 días (15 al 22); con la regla de 7 días el corte sería el 16/9. Se revisan los conteos diarios del final de la serie para decidir.

In [ ]:
# Conteos diarios de las últimas 4 semanas, comparados con el nivel de las 4 semanas previas
cola = diario.tail(28).to_frame()
cola['dia_semana'] = cola.index.day_name()
nivel_previo = diario.iloc[-56:-28].mean()
cola['ratio_vs_promedio_previo'] = (cola['transacciones'] / nivel_previo).round(2)
print(f'Promedio diario de las 4 semanas previas a la cola: {nivel_previo:,.0f}')
display(cola)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(cola.index, cola['transacciones'], color=AZUL)
ax.axhline(nivel_previo, color=ROJO, linestyle='--', label='Promedio de las 4 semanas previas')
ax.set_title('Transacciones diarias — últimas 4 semanas del dataset')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Definición del corte: test = últimos 7 días calendario del dataset
ULTIMO_DIA = transactions['t_dat'].max()
CORTE = ULTIMO_DIA - pd.Timedelta(days=6)   # primer día del test
print(f'Último día del dataset: {ULTIMO_DIA.date()}')
print(f'Test  : {CORTE.date()} a {ULTIMO_DIA.date()} (7 días)')
print(f'Train : hasta {(CORTE - pd.Timedelta(days=1)).date()}')

# Partición temporal
train = transactions[transactions['t_dat'] < CORTE]
test  = transactions[transactions['t_dat'] >= CORTE]

print('\nTAMAÑO DE LAS PARTICIONES')
print('=' * 70)
print(f'Train: {len(train):>10,} transacciones ({len(train) / len(transactions) * 100:.1f} %) | clientes: {train["customer_id"].nunique():,}')
print(f'Test : {len(test):>10,} transacciones ({len(test) / len(transactions) * 100:.1f} %) | clientes: {test["customer_id"].nunique():,}')

<a id="12"></a>
## 12. Clientes Fríos y Cobertura de Artículos

Se cuantifica cuántos clientes del test son **fríos naturales** (no tienen compras en train) y qué proporción de los artículos comprados en el test no aparece en train. Estas cifras se informan por separado de los fríos simulados, que se definirán ocultando historial (0/1/2 compras visibles).

In [ ]:
# Clientes fríos naturales: compran en test y no tienen ninguna compra en train
clientes_train = set(train['customer_id'].unique())
clientes_test  = set(test['customer_id'].unique())
frios_naturales = clientes_test - clientes_train
calientes = clientes_test & clientes_train

print('CLIENTES DEL TEST')
print('=' * 70)
print(f'Clientes que compran en test        : {len(clientes_test):,}')
print(f'  con historial en train (calientes): {len(calientes):,} ({len(calientes) / len(clientes_test) * 100:.1f} %)')
print(f'  sin historial (fríos naturales)   : {len(frios_naturales):,} ({len(frios_naturales) / len(clientes_test) * 100:.1f} %)')

trans_frios = test[test['customer_id'].isin(frios_naturales)]
print(f'Transacciones del test de fríos naturales: {len(trans_frios):,} ({len(trans_frios) / len(test) * 100:.1f} % del test)')

In [ ]:
# Cobertura de artículos: ¿los artículos comprados en test existen en train?
arts_train = set(train['article_id'].unique())
arts_test  = set(test['article_id'].unique())
ausentes_u = arts_test - arts_train
ausentes_t = (~test['article_id'].isin(arts_train)).sum()

print('COBERTURA DE ARTÍCULOS TEST vs TRAIN')
print('=' * 70)
print(f'Artículos distintos comprados en test: {len(arts_test):,}')
print(f'  ausentes en train (artículos nuevos): {len(ausentes_u):,} ({len(ausentes_u) / len(arts_test) * 100:.1f} % de los distintos)')
print(f'Transacciones del test sobre artículos ausentes en train: {ausentes_t:,} ({ausentes_t / len(test) * 100:.1f} %)')

# Referencia: peso de los 12 artículos más vendidos de train dentro del test
top12 = train['article_id'].value_counts().head(12).index
peso_top12 = test['article_id'].isin(top12).mean() * 100
print(f'\nPeso de los 12 artículos más vendidos de train dentro de las transacciones del test: {peso_top12:.2f} %')

In [ ]:
# Perfil de los fríos naturales vs. los clientes con historial (edad y club)
perfil_test = customers[customers['customer_id'].isin(clientes_test)].copy()
perfil_test['tipo'] = np.where(perfil_test['customer_id'].isin(frios_naturales), 'frío natural', 'con historial')

print('EDAD SEGÚN TIPO DE CLIENTE (test)')
display(perfil_test.groupby('tipo')['age'].describe())
print('club_member_status SEGÚN TIPO DE CLIENTE (test, %)')
display((pd.crosstab(perfil_test['tipo'], perfil_test['club_member_status'], normalize='index') * 100).round(1))

<a id="13"></a>
## 13. Hallazgos Consolidados y Decisiones Pendientes

Las cifras de este resumen se calculan directamente de las secciones anteriores. La redacción interpretativa de las conclusiones se completa una vez revisados los resultados reales.

In [ ]:
# Tablero de cifras clave (todas calculadas, ninguna escrita a mano)
resumen = pd.Series({
    'Transacciones (filas)': f'{len(transactions):,}',
    'Clientes': f'{customers.shape[0]:,}',
    'Artículos en catálogo': f'{articles.shape[0]:,}',
    'Artículos con al menos una venta': f'{transactions["article_id"].nunique():,}',
    'Rango de fechas': f'{fecha_min.date()} a {fecha_max.date()}',
    'Edad faltante (%)': f'{customers["age"].isnull().mean() * 100:.2f}',
    'Clientes con una sola transacción (%)': f'{(compras_cliente == 1).mean() * 100:.1f}',
    'Primer día del test (corte)': f'{CORTE.date()}',
    'Clientes fríos naturales en el test (%)': f'{len(frios_naturales) / len(clientes_test) * 100:.1f}',
    'Artículos del test ausentes en train (%)': f'{len(ausentes_u) / len(arts_test) * 100:.1f}',
}, name='valor')
display(resumen.to_frame())

**Decisiones pendientes a definir con el equipo:**
- Fecha de corte final (a partir de los conteos diarios de la sección 11).
- Tratamiento de la edad faltante y de las edades atípicas.
- Método de simulación de clientes fríos (0/1/2 compras visibles).
- Tratamiento de los artículos del test ausentes en train.